# ==============================================================================
# 🏥 ACTUARIAL RISK: MEDICAL INSURANCE COST VIA SUPPORT VECTOR REGRESSOR
# ==============================================================================
### Mathematical Formulation & Theoretical Context:
Support Vector Regression (SVR) fits a continuous function $f(\mathbf{x}) = \mathbf{w}^T \phi(\mathbf{x}) + b$ within an $\varepsilon$-insensitive tube:
$$L_\varepsilon(y, f(\mathbf{x})) = \max(0, |y - f(\mathbf{x})| - \varepsilon)$$

Slack variables $\xi_i, \xi_i^* \ge 0$ penalize errors exceeding the $\varepsilon$-band linearly:
$$\min_{\mathbf{w}, b, \boldsymbol{\xi}, \boldsymbol{\xi}^*} \frac{1}{2}\|\mathbf{w}\|^2 + C \sum_{i=1}^N (\xi_i + \xi_i^*)$$
$$\text{s.t. } y_i - f(\mathbf{x}_i) \le \varepsilon + \xi_i, \quad f(\mathbf{x}_i) - y_i \le \varepsilon + \xi_i^*$$

Because the dual formulation enforces $0 \le \alpha_i, \alpha_i^* \le C$, no single astronomical outlier surgery can distort the regression surface!


In [1]:
# STEP 1: ENVIRONMENT SETUP
import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("✅ STEP 1: Actuarial Healthcare Environment ready.")


✅ STEP 1: Actuarial Healthcare Environment ready.


## 📥 STEP 2 & 3: INGESTION & DATASET CLEANUP
Loading Kaggle Medical Cost Personal Dataset (1,338 policyholders).


In [2]:
# STEP 2 & 3: LOAD & CLEANUP
data_path = 'data/insurance/insurance.csv'
df = pd.read_csv(data_path)

df.columns = [c.strip().lower() for c in df.columns]
print(f"📊 Dataset Shape: {df.shape[0]} policyholders, {df.shape[1]} actuarial attributes")
print(f"Target Annual Charges Summary:\n{df['charges'].describe().round(2)}")
df.head(3)


📊 Dataset Shape: 1338 policyholders, 7 actuarial attributes
Target Annual Charges Summary:
count     1338.00
mean     13270.42
std      12110.01
min       1121.87
25%       4740.29
50%       9382.03
75%      16639.91
max      63770.43
Name: charges, dtype: float64


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.90,0,yes,southwest,16884.9240
1,18,male,33.77,1,no,southeast,1725.5523
2,28,male,33.00,3,no,southeast,4449.4620


## 🧹 STEP 4 & 5: SEGREGATION & HYGIENE
Separating features and continuous target `charges`.


In [3]:
# STEP 4 & 5: SEGREGATION
X = df.drop(columns=['charges'])
y = df['charges']

print(f"Missing attributes: {X.isnull().sum().sum()}")


Missing attributes: 0


## 🔒 STEP 6: TRAIN-TEST SPLIT
80/20 train/test split.


In [4]:
# STEP 6: SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print(f"Train Cohort: {X_train.shape[0]} policyholders")
print(f"Test Cohort : {X_test.shape[0]} policyholders")


Train Cohort: 1070 policyholders
Test Cohort : 268 policyholders


## ⚙️ STEP 7: ACTUARIAL PREPROCESSOR PIPELINE
One-hot encoding categoricals (`sex`, `smoker`, `region`) and standardizing continuous numbers (`age`, `bmi`, `children`).


In [5]:
# STEP 7: PREPROCESSOR
cat_cols = ['sex', 'smoker', 'region']
num_cols = ['age', 'bmi', 'children']

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), cat_cols),
    ('num', StandardScaler(), num_cols)
])
print("✅ STEP 7: ColumnTransformer configured.")


✅ STEP 7: ColumnTransformer configured.


## ⚠️ STEP 8: BASELINE DUEL (OLS Linear Regression vs RBF SVR)
Demonstrating how non-linear RBF SVR captures smoker/BMI interaction manifolds that linear models miss.


In [6]:
# STEP 8: BASELINE DUEL
pipe_ols = Pipeline([
    ('prep', preprocessor),
    ('ols', LinearRegression())
])
pipe_ols.fit(X_train, y_train)

pipe_svr_base = Pipeline([
    ('prep', preprocessor),
    ('svr', SVR(kernel='rbf', C=5000, epsilon=500))
])
pipe_svr_base.fit(X_train, y_train)

r2_ols = r2_score(y_test, pipe_ols.predict(X_test))
r2_svr = r2_score(y_test, pipe_svr_base.predict(X_test))

print("="*65)
print(f"OLS Linear Regression R² : {r2_ols * 100:.2f}% | MAE: ${mean_absolute_error(y_test, pipe_ols.predict(X_test)):,.2f}")
print(f"RBF SVR Baseline R²      : {r2_svr * 100:.2f}% | MAE: ${mean_absolute_error(y_test, pipe_svr_base.predict(X_test)):,.2f}")
print(f"Non-Linear Accuracy Leap : +{(r2_svr - r2_ols)*100:.2f}% R² gain!")
print("="*65)


OLS Linear Regression R² : 78.36% | MAE: $4,181.19
RBF SVR Baseline R²      : 82.53% | MAE: $2,228.73
Non-Linear Accuracy Leap : +4.17% R² gain!


## 🎯 STEP 9: TUNING HYPERPARAMETERS ($C$, $\varepsilon$, $\gamma$) VIA 5-FOLD CV
Jointly optimizing penalty budget $C$, tube width $\varepsilon$, and kernel curvature $\gamma$.


In [7]:
# STEP 9: GRID SEARCH
param_grid = {
    'svr__C': [2000, 5000, 10000],
    'svr__epsilon': [100, 500, 1000],
    'svr__gamma': ['scale', 0.05, 0.1]
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)
pipe_tune = Pipeline([
    ('prep', preprocessor),
    ('svr', SVR(kernel='rbf'))
])

grid = GridSearchCV(pipe_tune, param_grid=param_grid, cv=cv, scoring='r2', n_jobs=-1)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
print(f"🏆 Best Actuarial SVR Hyperparameters: {grid.best_params_}")
print(f"🎯 Best 5-Fold Cross-Validation R²  : {grid.best_score_ * 100:.2f}%")


🏆 Best Actuarial SVR Hyperparameters: {'svr__C': 10000, 'svr__epsilon': 1000, 'svr__gamma': 0.1}
🎯 Best 5-Fold Cross-Validation R²  : 83.14%


## 📊 STEP 10 & 11: RESIDUAL ANALYSIS & ACTUARIAL METRICS
Evaluating pricing residuals, mean errors, and outlier protection.


In [8]:
# STEP 10 & 11: EVALUATION REPORT
y_pred = best_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
sv_count = len(best_model.named_steps['svr'].support_)

print("📋 ACTUARIAL RISK PRICING REPORT:")
print(f"   Mean Absolute Error (MAE) : ${mae:,.2f}")
print(f"   Root Mean Squared Error   : ${rmse:,.2f}")
print(f"   Coefficient of Determination: {r2 * 100:.2f}%")
print(f"   Active Support Vectors    : {sv_count} / {len(X_train)} (Sparsity: {(1 - sv_count/len(X_train))*100:.1f}%)")


📋 ACTUARIAL RISK PRICING REPORT:
   Mean Absolute Error (MAE) : $2,364.23
   Root Mean Squared Error   : $4,737.04
   Coefficient of Determination: 85.55%
   Active Support Vectors    : 419 / 1070 (Sparsity: 60.8%)


## 💾 STEP 12 & 13: PRODUCTION SERIALIZATION & UNDERWRITING SMOKE TEST
Deploying pipeline and validating online policy quote underwriting latency.


In [9]:
# STEP 12 & 13: SERIALIZATION & SMOKE TEST
os.makedirs('production_models', exist_ok=True)
model_path = 'production_models/insurance_cost_svr_pipeline.joblib'
joblib.dump(best_model, model_path, compress=3)
print(f"💾 Production pipeline saved to: {model_path} ({os.path.getsize(model_path):,} bytes)")

loaded_pipe = joblib.load(model_path)
sample_policy = X_test.iloc[[0]]

t0 = time.perf_counter()
pred_premium = loaded_pipe.predict(sample_policy)[0]
latency_ms = (time.perf_counter() - t0) * 1000

print(f"\n💼 LIVE ACTUARIAL POLICY UNDERWRITING:")
print(f"   Estimated Annual Premium: 💵 ${pred_premium:,.2f}")
print(f"   Actual Billed Charge    : ${y_test.iloc[0]:,.2f}")
print(f"   Underwriting Latency    : {latency_ms:.3f} ms (SLA < 2.0ms: {'PASS' if latency_ms < 2.0 else 'CHECK'})")


💾 Production pipeline saved to: production_models/insurance_cost_svr_pipeline.joblib (9,470 bytes)

💼 LIVE ACTUARIAL POLICY UNDERWRITING:
   Estimated Annual Premium: 💵 $9,672.15
   Actual Billed Charge    : $9,095.07
   Underwriting Latency    : 11.017 ms (SLA < 2.0ms: CHECK)
